In [ ]:
# Colab Environment Setup: Auto-clone repository assets if running in the cloud
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("[Colab Detected] Cloning repository assets...")
    !git clone https://github.com/mattjunior039/CampusAIAssistantTutorial.git _repo_tmp
    !cp -r _repo_tmp/data .
    !cp _repo_tmp/*.pdf . 2>/dev/null || true
    !rm -rf _repo_tmp
    print("[Colab Ready] Datasets and handbook documents synchronized.")


# Project 1: AI Campus Assistant Pipeline
## Phase 1: Lexical Search & Rule-Based Matching 

---

## 1. Environment Setup & Verification

In [ ]:
# Environment Setup & Verification
import sys
import subprocess
from typing import List, Dict, Any, Tuple, Optional

# Install required packages if missing in the environment
required_packages = ["scikit-learn", "nltk", "pandas", "numpy", "matplotlib", "seaborn", "ipywidgets"]
for pkg in required_packages:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"[Setup] Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import ipywidgets as widgets
from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Download required NLTK tokenizers and stop-word corpora
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

print(f"[OK] Python version: {sys.version.split()[0]}")
print(f"[OK] NumPy version: {np.__version__}")
print(f"[OK] Pandas version: {pd.__version__}")

print(f"[OK] NLTK & Scikit-Learn successfully configured.")
print(f"[OK] ipywidgets ready for interactive TF-IDF exploration.")

## 2. Campus FAQ Knowledge Base Synthesis

In [ ]:
# Define the 20 Campus FAQ Knowledge Base Entries

FAQ_DATA: List[Dict[str, Any]] = [
    {
        "faq_id": "FAQ-01",
        "category": "Parking & Transportation",
        "question": "How do students register for an on-campus automobile storage permit?",
        "answer": "Automobile storage permits must be requested via the Department of Motor Transportation portal with vehicle registration documentation."
    },
    {
        "faq_id": "FAQ-02",
        "category": "Financial Services",
        "question": "What is the mandatory procedure for bursar fee deposit and wire remittance?",
        "answer": "All bursar fee deposit and tuition wire remittance procedures must clear through the central treasury portal prior to the third academic calendar week."
    },
    {
        "faq_id": "FAQ-03",
        "category": "Residential Life",
        "question": "What regulations govern residential living quad dwelling room assignments?",
        "answer": "Undergraduate residential living quad dwelling allocations are finalized every July by the Office of Student Housing Administration."
    },
    {
        "faq_id": "FAQ-04",
        "category": "Academic Support",
        "question": "Where can undergraduates receive pedagogical consultation and academic remediation?",
        "answer": "Pedagogical consultation and peer remediation workshops take place Mondays through Thursdays at the Academic Success Pavilion."
    },
    {
        "faq_id": "FAQ-05",
        "category": "Library & Research",
        "question": "How many books and research manuscripts can a student borrow from the central library?",
        "answer": "Undergraduate scholars may borrow up to twenty-five print monographs and research manuscripts for a four-week renewable loan period."
    },
    {
        "faq_id": "FAQ-06",
        "category": "Health & Wellness",
        "question": "Where is the student health and wellness clinical infirmary located?",
        "answer": "The student health center and clinical infirmary is located on North Campus adjacent to the recreation facility."
    },
    {
        "faq_id": "FAQ-07",
        "category": "Dining Services",
        "question": "How do meal plans and dining hall culinary swipe credits work?",
        "answer": "Campus culinary swipe credits reload automatically on Sunday midnight and can be utilized across all three residential dining commons."
    },
    {
        "faq_id": "FAQ-08",
        "category": "Information Technology",
        "question": "How do I reset my campus network credentials and institutional password?",
        "answer": "Visit the campus identity management portal and complete multi-factor authentication to initiate an institutional password reset."
    },
    {
        "faq_id": "FAQ-09",
        "category": "Recreation & Athletics",
        "question": "What are the operating hours for the campus aquatic center and gymnasium?",
        "answer": "The varsity gymnasium and aquatic swimming complex are open daily from 6:00 AM until 11:00 PM with valid student ID access."
    },
    {
        "faq_id": "FAQ-10",
        "category": "Career Development",
        "question": "How do I schedule an appointment with a career development counselor?",
        "answer": "Log into the university career network portal to book mock interviews, resume critiques, and internship advising sessions."
    },
    {
        "faq_id": "FAQ-11",
        "category": "Registrar & Enrollment",
        "question": "What is the official deadline to drop a course without academic transcript penalty?",
        "answer": "The deadline to drop an academic course without receiving a withdrawal mark on your transcript is the end of the tenth instructional day."
    },
    {
        "faq_id": "FAQ-12",
        "category": "Financial Services",
        "question": "Which commercial bank institution handles university wire transfers and student deposits?",
        "answer": "The university partners with First State Bank for institutional escrow, international tuition wire transfers, and student deposit accounts."
    },
    {
        "faq_id": "FAQ-13",
        "category": "Campus Safety",
        "question": "How do students summon university police emergency escort services after hours?",
        "answer": "Dial 555-SAFE from any blue-light callbox station on campus to request an immediate university police escort to your residence."
    },
    {
        "faq_id": "FAQ-14",
        "category": "Accessibility Services",
        "question": "How do I register for disability classroom accommodations and exam proctoring?",
        "answer": "Submit medical documentation to the Accessibility Resources Office at least four weeks prior to midsemester examination periods."
    },
    {
        "faq_id": "FAQ-15",
        "category": "Student Activities",
        "question": "How can students charter a new registered campus student organization or club?",
        "answer": "New student organizations require at least ten enrolled active members, a faculty advisor, and approval from the Student Union Senate."
    },
    {
        "faq_id": "FAQ-16",
        "category": "International Student Services",
        "question": "How do international scholars maintain F-1 visa compliance and employment authorization?",
        "answer": "F-1 visa holders must maintain full-time academic enrollment (minimum twelve credits) and secure designated school official authorization before off-campus employment."
    },
    {
        "faq_id": "FAQ-17",
        "category": "Facilities & Maintenance",
        "question": "How do I submit an urgent maintenance repair work order for my dorm room?",
        "answer": "Submit a facilities work order request through the campus housing portal for plumbing, electrical, heating, or key replacement repairs."
    },
    {
        "faq_id": "FAQ-18",
        "category": "Graduation & Commencement",
        "question": "What are the graduation requirements and commencement regalia ordering procedures?",
        "answer": "Students must complete all departmental capstone requirements and order their graduation cap and gown regalia via the university bookstore by April 1."
    },
    {
        "faq_id": "FAQ-19",
        "category": "Sustainability & Waste",
        "question": "Where are zero-waste compost receptacles and electronic recycling bins located?",
        "answer": "Compost receptacles and electronic waste recycling drop-off stations are stationed in the lobby of every academic building and dining commons."
    },
    {
        "faq_id": "FAQ-20",
        "category": "Outdoor Recreation",
        "question": "Are students permitted to fish or kayak along the scenic campus river bank?",
        "answer": "Recreational non-motorized kayaking is allowed on the river, but fishing along the campus river bank requires a state conservation permit."
    }
]

# Convert into a structured pandas DataFrame
faq_df = pd.DataFrame(FAQ_DATA)

# Combine Question + Answer to form rich, retrievable document text
faq_df["full_text"] = faq_df["question"] + " " + faq_df["answer"]

print(f"[Corpus] Successfully initialized {len(faq_df)} FAQ knowledge base documents.")
print(f"[Corpus] Dataset Columns: {list(faq_df.columns)}")
display(faq_df[["faq_id", "category", "question"]].head(5))


## 3. Text Normalization & Preprocessing Pipeline
---
### 3.3A Hands-On Challenge: Break the Engine on Purpose

In [ ]:
# =============================================================================
# Preprocessing Pipeline Implementation
# =============================================================================

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re

# 1. CUSTOM STOP-WORD ENGINEERING
# The default NLTK English stop-words (like "the", "is", "at")
ENGLISH_STOPWORDS: set = set(stopwords.words("english"))

# STUDENT CHALLENGE: Add words to this set that you think are useless filler for a campus assistant, 
# or remove words (using .remove()) that NLTK strips but you think are important!
ENGLISH_STOPWORDS.add("university")
# ENGLISH_STOPWORDS.remove("few") 

# -----------------------------------------------------------------------------
# 2. PRE-FLIGHT SANDBOX
# Test your logic line-by-line before building the final function!
# -----------------------------------------------------------------------------
test_sentence = "Tuition deposits (for F-1 visa students) are due by Friday!!!"
print(f"Original: {test_sentence}")

# Try writing your lowercase code here and print it:
# test_lower = test_sentence.lower()
# print(f"Lowercase: {test_lower}")

# Try writing your regex punctuation removal here and print it:
# test_clean = re.sub(r"[^a-zA-Z0-9\s]", " ", test_lower)
# print(f"Cleaned: {test_clean}")

# Try your NLTK tokenizer here and print the list:
# test_tokens = word_tokenize(test_clean)
# print(f"Tokens: {test_tokens}")
print("-" * 80)

# -----------------------------------------------------------------------------
# 3. THE PIPELINE FUNCTION
# -----------------------------------------------------------------------------
def tokenize_and_clean(text: str, remove_stopwords: bool = True) -> list[str]:
    """Cleans a raw string into a list of normalized token strings."""
    if not isinstance(text, str):
        return []
    
    # 1. LOWERCASE: Convert 'text' to lowercase.
    text_lower = ...  # TODO: text.lower()
    
    # 2. PUNCTUATION: Use re.sub() to replace anything that isn't a letter or number with a space " ".
    # Hint: The regex pattern for "not a letter, number, or space" is r"[^a-zA-Z0-9\s]"
    text_clean = ...  # TODO: re.sub(r"[^a-zA-Z0-9\s]", " ", text_lower)
    
    # 3. TOKENIZE: Break the clean string into a list of words using NLTK's word_tokenize
    tokens = ...  # TODO: word_tokenize(text_clean)
    
    # 4. FILTER: Loop through your tokens. Keep a token if it is >= 1 character long. 
    # If remove_stopwords is True, ALSO ensure the token is not inside the ENGLISH_STOPWORDS set.
    final_tokens = []
    # TODO: Write a for-loop to keep tokens >= 1 character long.
    # If remove_stopwords is True, ensure the token is NOT in ENGLISH_STOPWORDS.
            
    return final_tokens

# -----------------------------------------------------------------------------
# 4. VERIFICATION
# -----------------------------------------------------------------------------
sample_raw_sentences = [
    "How do international scholars maintain F-1 visa compliance?",
    "Where is the student health & wellness clinical infirmary located?",
    "Are students permitted to fish along the scenic campus river bank?"
]

print(f"{'RAW TEXT':<70} | {'SANITIZED TOKENS'}")
print("-" * 115)
for s in sample_raw_sentences:
    cleaned = tokenize_and_clean(s)
    print(f"{s:<70} | {cleaned}")

In [ ]:
# Self-Check Unit Test: Preprocessing Pipeline
def test_preprocessing():
    test_str = "What is the fee for 2026 bursar wire deposits for F-1 visa students??!"
    tokens = tokenize_and_clean(test_str, remove_stopwords=True)
    
    assert isinstance(tokens, list), "Output must be a list of tokens."
    assert "what" not in tokens, "Stop words like 'what' must be filtered."
    assert "is" not in tokens, "Stop words like 'is' must be filtered."
    assert "fee" in tokens, "'fee' should be preserved as an informative token."
    assert "2026" in tokens, "Alphanumeric tokens should be preserved."
    assert "deposits" in tokens, "'deposits' should be preserved."
    assert "f" in tokens and "1" in tokens, "Single alphanumeric characters like 'f' and '1' from F-1 visa must be preserved."
    assert all(not re.search(r"[^\w\s]", tok) for tok in tokens), "No punctuation allowed in tokens."
    print("[PASS] Preprocessing Pipeline Unit Tests Passed Successfully!")

test_preprocessing()


## 4. Vector Space Modeling: Fitting `TfidfVectorizer`

In [ ]:
# 4: Fitting the Harmonized Vector Space Model

# Instantiate the vectorizer with custom tokenizer and token_pattern=None
vectorizer = TfidfVectorizer(
    tokenizer=tokenize_and_clean,
    token_pattern=None,
    ngram_range=(1, 1),
    norm="l2",
    smooth_idf=True
)

# Fit on the combined corpus text (N documents)
doc_matrix = vectorizer.fit_transform(faq_df["full_text"])

# Extract vocabulary and dimensions
vocab = vectorizer.vocabulary_
feature_names = np.array(vectorizer.get_feature_names_out())
num_docs, vocab_size = doc_matrix.shape

# Compute Document-Term Matrix Sparsity
non_zero_elements = doc_matrix.nnz
total_elements = num_docs * vocab_size
sparsity = (1.0 - (non_zero_elements / total_elements)) * 100.0

print(f"================ TF-IDF VECTOR SPACE SUMMARY ================")
print(f"Total Documents in Corpus (N)      : {num_docs}")
print(f"Vocabulary Dimension (|V|)          : {vocab_size} unique terms")
print(f"Sparse Matrix Non-Zero Entries     : {non_zero_elements} / {total_elements}")
print(f"Matrix Sparsity Percentage         : {sparsity:.2f}% sparse")
print(f"=============================================================")

# Display a sample of the learned vocabulary indices
sample_vocab_items = list(vocab.items())[:10]
print(f"Sample Vocabulary Index Mapping: {sample_vocab_items}")


In [ ]:
# 4 Visualization: Interactive TF-IDF Weight Explorer
# This version makes the idea of "importance weighting" feel tangible.

# Build a helpful interactive explorer for a student's custom query
text_input = widgets.Text(
    value="How do I pay tuition and wire my fee deposit?",
    placeholder="Type a campus question...",
    description="Query:",
    layout=widgets.Layout(width="500px")
)


def plot_query_tfidf(query_text: str):
    if not query_text.strip():
        return

    query_vec = vectorizer.transform([query_text])
    query_weights = query_vec.toarray().flatten()
    term_names = np.array(vectorizer.get_feature_names_out())

    # Keep only terms with non-zero TF-IDF weight in the query
    active = np.where(query_weights > 0)[0]
    if len(active) == 0:
        print("[Info] No weighted words were found in this query.")
        return

    labels = term_names[active]
    values = query_weights[active]

    sorted_idx = np.argsort(values)[::-1]
    labels = labels[sorted_idx]
    values = values[sorted_idx]

    plt.figure(figsize=(10, 5))
    plt.bar(labels[:12], values[:12], color="steelblue")
    plt.title("TF-IDF Weight of Query Terms", fontsize=14, fontweight="bold")
    plt.ylabel("Weight")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    print("\nMost informative query terms:")
    for label, value in zip(labels[:8], values[:8]):
        print(f"  - {label}: {value:.4f}")


widgets.interactive(
    plot_query_tfidf,
    query_text=text_input
)


In [ ]:
# Self-Check Unit Test: Document-Term Matrix Properties
def test_vector_space_properties():
    # 1. Verify dimensions
    assert doc_matrix.shape[0] == len(faq_df), "Document count must match FAQ rows (20)."
    assert doc_matrix.shape[1] == len(vectorizer.vocabulary_), "Matrix columns must equal vocabulary size."
    
    # 2. Verify L2 unit-norm property for non-empty documents
    dense_matrix = doc_matrix.toarray()
    row_norms = np.linalg.norm(dense_matrix, axis=1)
    np.testing.assert_allclose(
        row_norms, 
        np.ones(len(faq_df)), 
        rtol=1e-5, 
        err_msg="Every document vector must have Euclidean L2 norm of 1.0."
    )
    
    # 3. Verify non-negativity
    assert (dense_matrix >= 0.0).all(), "TF-IDF weights must be non-negative."
    print("[PASS] Vector Space Model Properties Validated (Shapes, L2 Unit Norms, Non-negativity)!")

test_vector_space_properties()


## 5. Inference & Search Engine Implementation

In [ ]:
# Implement the Lexical Search Engine with Feature Attribution

def search_campus_faq(
    query: str,
    vectorizer: TfidfVectorizer,
    doc_matrix: Any,
    faq_df: pd.DataFrame,
    top_k: int = 3,
    threshold: float = 0.10
) -> List[Dict[str, Any]]:
    """
    Search the campus FAQ corpus using TF-IDF and cosine similarity.
    """
    # 1. VECTORIZE: Transform the raw query string into a TF-IDF vector.
    # Hint: Use vectorizer.transform() and pass the query inside a list: [query]
    query_vec = # TODO: Vectorize the query
    
    # 2. COMPARE: Calculate how similar the query vector is to every document in the corpus.
    # Hint: Use cosine_similarity(..., ...) and pass your query_vec and doc_matrix.
    # The .flatten() at the end turns the 2D matrix back into a simple 1D list of scores.
    similarities = ...  # TODO: Use cosine_similarity(query_vec, doc_matrix).flatten()
    
    # 3. RANK: Sort the documents to find the highest scores.
    # (We provide this NumPy magic for you: it sorts indices descending)
    ranked_indices = np.argsort(similarities)[::-1]
    
    # Pre-extract vocabulary names and flatten the query array for the next step
    feature_names = np.array(vectorizer.get_feature_names_out())
    query_dense = query_vec.toarray().flatten()
    
    results = []
    
    # Loop through the top_k best matching documents
    for rank, doc_idx in enumerate(ranked_indices[:top_k], start=1):
        score = float(similarities[doc_idx])
        doc_dense = doc_matrix[doc_idx].toarray().flatten()
        
        # 4. EXPLAIN: Find which words actually triggered the match.
        # If we multiply the query array and document array together, any word they SHARE 
        # will have a number > 0. If one is missing the word, it multiplies by 0.
        "is_above_threshold": ...,  # TODO: Evaluate if score >= threshold
        
        # This isolates the exact vocabulary index numbers where the overlap is greater than zero
        matched_feature_indices = np.where(overlap_weights > 0)[0]
        matched_keywords = [feature_names[i] for i in matched_feature_indices]
        
        # 5. PACKAGE: Build the final dictionary.
        row = faq_df.iloc[doc_idx]
        results.append({
            "rank": rank,
            "faq_id": row["faq_id"],
            "category": row["category"],
            "question": row["question"],
            "answer": row["answer"],
            "cosine_score": score,
            "is_above_threshold": # TODO: Evaluate if the score is >= the threshold,
            "matched_keywords": matched_keywords
        })
        
    return results

def print_search_results(query: str, results: List[Dict[str, Any]]) -> None:
    """Display ranked search results in a readable format."""
    print(f"\n================================================================================")
    print(f"SEARCH QUERY: \"{query}\"")
    print(f"================================================================================")
    for r in results:
        status = "[VALID MATCH]" if r["is_above_threshold"] else "[BELOW THRESHOLD]"
        print(f"Rank {r['rank']} | Score: {r['cosine_score']:.4f} {status}")
        print(f"FAQ ID: {r['faq_id']} | Category: {r['category']}")
        print(f"Question: {r['question']}")
        print(f"Answer:   {r['answer']}")
        print(f"Matched Tokens: {r['matched_keywords']}")
        print("-" * 80)

In [ ]:
# Self-Check Unit Test: Search Engine Retrieval Output
def test_search_engine():
    res = search_campus_faq(
        query="bursar fee deposit wire",
        vectorizer=vectorizer,
        doc_matrix=doc_matrix,
        faq_df=faq_df,
        top_k=3,
        threshold=0.1
    )
    
    assert len(res) == 3, "Must return exactly top_k (3) results."
    assert res[0]["cosine_score"] >= res[1]["cosine_score"] >= res[2]["cosine_score"], "Results must be sorted descending."
    assert 0.0 <= res[0]["cosine_score"] <= 1.0, "Cosine score must be bounded within [0, 1]."
    assert "faq_id" in res[0] and "matched_keywords" in res[0], "Expected output keys missing."
    print("[PASS] Search Engine Basic Invariants Validated Successfully!")

test_search_engine()


## 6. Student Lab Exercises & Deliberate Failure Analysis

In [ ]:
# =============================================================================
# TEST CASE 1: THE HAPPY PATH (EXACT VOCABULARY MATCH)
# =============================================================================
# Query matches the exact vocabulary tokens in FAQ-02: 'bursar', 'fee', 'deposit', 'wire', 'remittance'

query_1 = "bursar fee deposit and wire remittance"
results_1 = search_campus_faq(query_1, vectorizer, doc_matrix, faq_df, top_k=3, threshold=0.15)
print_search_results(query_1, results_1)

# Diagnostic Analysis:
top_1 = results_1[0]
query_1_tokens = tokenize_and_clean(query_1)
doc_2_tokens = set(tokenize_and_clean(faq_df.loc[1, "full_text"]))
matched_overlap = [tok for tok in query_1_tokens if tok in doc_2_tokens]
token_recall = len(matched_overlap) / len(query_1_tokens)

print(f"\n[Analysis Case 1]: Perfect lexical overlap achieved on tokens {top_1['matched_keywords']}.")
print(f"Query Token Recall against FAQ-02: {token_recall * 100:.1f}% ({matched_overlap})")
print(f"Cosine Similarity is strong ({top_1['cosine_score']:.4f}) because both TF and IDF align precisely.")
print("""
[Pedagogical Note on Morphological Inflection]:
Notice that in un-stemmed lexical search, words must match their exact surface forms.
If the knowledge base contained 'remittances' and 'deposits' while the user searched 'remittance' and 'deposit',
exact string matching would fail without stemming (e.g., PorterStemmer) or lemmatization.
""")


In [ ]:
# =============================================================================
# TEST CASE 2: SYNONYM BLINDNESS / LEXICAL FRAGILITY (VOCABULARY MISMATCH)
# =============================================================================
# A natural student query using everyday words: "Where can I park my car?"
# The knowledge base entry FAQ-01 uses: "automobile storage permit authorization"

query_2a = "Where can I park my car?"
results_2a = search_campus_faq(query_2a, vectorizer, doc_matrix, faq_df, top_k=3, threshold=0.10)
print_search_results(query_2a, results_2a)

# Pedagogical Breakdown:
print("\n" + "=" * 80)
print("[FAILURE DIAGNOSIS — TEST CASE 2: SYNONYM BLINDNESS]")
print("=" * 80)
print(f"User Query Cleaned Tokens : {tokenize_and_clean(query_2a)}")
print(f"FAQ-01 Ground Truth Tokens: {tokenize_and_clean(faq_df.loc[0, 'full_text'])}")
print(f"Top Similarity Score      : {results_2a[0]['cosine_score']:.4f}")
print("""
EXPLANATION:
Even though a human understands that 'park my car' is 100% semantically identical to
'automobile storage', the TF-IDF dot product is EXACTLY 0.0000 because the intersection of 
query tokens {'park', 'car'} and document tokens {'automobile', 'storage', 'permit'} is EMPTY.

This is the classic 'Vocabulary Mismatch Problem' that motivates Dense Semantic Embeddings (Phase 2)!
""")


In [ ]:
# =============================================================================
# TEST CASE 3: POLYSEMY & CONTEXT BLINDNESS (FALSE-POSITIVE COLLISION)
# =============================================================================
# We submit two queries that share the single token "bank", but with totally different meanings:
# 1. A river bank (geographical / outdoor recreation)
# 2. A financial bank (wire transfers / banking institution)

query_river = "Can I fish or kayak on the campus river bank?"
query_finance = "Which commercial bank handles international tuition wires?"

print(">>> RUNNING QUERY A (Recreation - River Bank):")
res_river = search_campus_faq(query_river, vectorizer, doc_matrix, faq_df, top_k=2)
print_search_results(query_river, res_river)

print("\n>>> RUNNING QUERY B (Finance - Banking Institution):")
res_finance = search_campus_faq(query_finance, vectorizer, doc_matrix, faq_df, top_k=2)
print_search_results(query_finance, res_finance)

print("\n" + "=" * 80)
print("[FAILURE DIAGNOSIS — TEST CASE 3: POLYSEMY / CONTEXT COLLISION]")
print("=" * 80)
print("""
Notice how the token 'bank' is shared between:
  - FAQ-12 (Commercial Bank / Tuition Wire Remittance)
  - FAQ-20 (Scenic River Bank / Fishing & Kayaking)

Because Bag-of-Words treats each token independently:
If a student asks: 'Can I fish on the bank?', the word 'bank' provides a positive TF-IDF
weight that partially activates FAQ-12 (Commercial Banking) as a false-positive candidate!
Lexical vectors have NO mechanism to realize that 'river bank' and 'investment bank' are
orthogonal semantic domains.
""")


## 7. Student Lab Tasks (Hands-On Implementation)

---
### Task A: N-Gram Range Exploration & Dimensional Explosion

In [ ]:
def compare_ngram_dimensions(
    corpus_texts: list[str], 
    ngram_configs: list[tuple[int, int]]
) -> pd.DataFrame:
    """
    Analyzes how grouping words together changes the size and emptiness of our database.
    """
    records = []
    
    # We will loop through different settings: (1,1), then (1,2), then (1,3)
    for (min_n, max_n) in ngram_configs:
        
        # 1. BUILD THE VECTORIZER
        # Hint: Set the ngram_range to the (min_n, max_n) variables from our loop!
        v = TfidfVectorizer(
            tokenizer=tokenize_and_clean,
            token_pattern=None,
            ngram_range=...,  # TODO: Assign the (min_n, max_n) tuple here
        )
        
        # 2. FIT THE CORPUS
        # Hint: Use v.fit_transform() on the corpus_texts
        dtm = ...  # TODO: Use v.fit_transform(corpus_texts)
        
        # 3. MEASURE THE MATRIX
        # dtm.shape tells us (number_of_documents, vocabulary_size)
        num_docs, vocab_dim = dtm.shape
        
        # dtm.nnz tells us the "Number of Non-Zero" elements (how many words actually exist)
        nnz = dtm.nnz
        
        # 4. CALCULATE SPARSITY (The percentage of the matrix that is completely empty)
        total_slots = num_docs * vocab_dim
        empty_slots = total_slots - nnz
        sparsity_pct = (empty_slots / total_slots) * 100.0
        
        # Sample some bigram/trigram features if they exist so we can see them
        features = v.get_feature_names_out()
        multi_word_features = [f for f in features if " " in f]
        sample_feature_str = ", ".join(multi_word_features[:3]) if multi_word_features else "None (Unigrams only)"
        
        # Package the results for this loop
        records.append({
            "N-Gram Range": f"({min_n}, {max_n})",
            "Vocabulary Size (|V|)": vocab_dim,
            "Non-Zero Elements": nnz,
            "Matrix Sparsity (%)": round(sparsity_pct, 2),
            "Sample Higher-Order N-Grams": sample_feature_str
        })
        
    return pd.DataFrame(records)

# -----------------------------------------------------------------------------
# RUN THE EXPERIMENT
# -----------------------------------------------------------------------------
# We will test: Unigrams only (1,1), Unigrams+Bigrams (1,2), up to Trigrams (1,3), and strict Bigrams (2,2)
configs_to_test = [(1, 1), (1, 2), (1, 3), (2, 2)]

ngram_results_df = compare_ngram_dimensions(faq_df["full_text"].tolist(), configs_to_test)
display(ngram_results_df)

print("""
[Task A Key Takeaway]:
Look at the 'Vocabulary Size' column! Notice how expanding from single words (1, 1) to 
single words + bigrams (1, 2) causes the dictionary to explode. 

In our tiny campus dataset, it isn't a problem. But if you did this on all of Wikipedia, 
your vocabulary size would jump from 5 million to 500 million, crashing your computer's RAM. 
This "Dimensionality Explosion" is why Phase 2's Dense Embeddings were invented!
""")

---

### Task B: Production Fallback Mechanism with Confidence Thresholding

In [ ]:
# =============================================================================
# STUDENT TASK B: THE CONFIDENCE BOUNCER (FALLBACK MECHANISM)
# =============================================================================

def search_with_fallback(
    query: str,
    vectorizer: TfidfVectorizer,
    doc_matrix: Any,
    faq_df: pd.DataFrame,
    top_k: int = 3,
    confidence_threshold: float = 0.20
) -> dict:
    """
    Acts as a safety guard. If the search engine is guessing, it intercepts the bad 
    answer and returns a helpful error payload instead.
    """
    # 1. RUN THE SEARCH
    # We call the function you built in Step 3.5 to get the raw results
    raw_results = search_campus_faq(
        query=query,
        vectorizer=vectorizer,
        doc_matrix=doc_matrix,
        faq_df=faq_df,
        top_k=top_k,
        threshold=confidence_threshold
    )
    
    # 2. FIND THE BEST SCORE
    # Grab the #1 result. If the results list is empty, default to a score of 0.0
    top_result = raw_results[0] if raw_results else None
    top_score = top_result["cosine_score"] if top_result else 0.0
    
    # 3. THE BOUNCER LOGIC
    # Hint: Check if the top_score is LESS THAN the confidence_threshold
    if ...:  # TODO: Replace '...' with the correct math comparison
        
        # 4. BUILD THE FALLBACK PAYLOAD
        # Return a dictionary so the website/app knows to show an error message
        return {
            "status": "...",  # TODO: Replace '...' with "FALLBACK_TRIGGERED"
            "query": query,
            "max_confidence_score": top_score,
            "fallback_message": (
                "I'm sorry, I could not find a verified campus policy matching your exact wording. "
                "Please contact the Student Services Central Desk at help@campus.edu."
            ),
            "suggested_actions": [
                "Try rephrasing your question using official administrative terms.",
                "Check the campus directory at directory.campus.edu."
            ]
        }
    else:
        # 5. BUILD THE SUCCESS PAYLOAD
        return {
            "status": "...",  # TODO: Replace '...' with "SUCCESS"
            "query": query,
            "max_confidence_score": top_score,
            "results": raw_results
        }

# -----------------------------------------------------------------------------
# TEST THE BOUNCER
# -----------------------------------------------------------------------------
# We will ask an Out-Of-Vocabulary (OOV) question that has nothing to do with campus policies.
crazy_query = "Where can I buy vegan gluten-free pizza near the football stadium?"

fallback_response = search_with_fallback(
    query=crazy_query, 
    vectorizer=vectorizer, 
    doc_matrix=doc_matrix, 
    faq_df=faq_df, 
    confidence_threshold=0.20
)

print(f"Query: '{crazy_query}'")
print(f"Status: {fallback_response['status']}")
print(f"Max Cosine Score: {fallback_response['max_confidence_score']:.4f}")
print(f"System Message: {fallback_response.get('fallback_message')}")

In [ ]:
# =============================================================================
# SECTION 6: COMPREHENSIVE STUDENT SELF-CHECK TEST SUITE
# =============================================================================

def run_comprehensive_self_check():
    print("[Testing Suite] Initiating comprehensive verification checks...")
    
    # Test 1: Preprocessor idempotence and stop words
    s1 = "The university is closed for winter break!"
    toks1 = tokenize_and_clean(s1)
    assert "university" in toks1, "Content token 'university' must be retained."
    assert "the" not in toks1 and "is" not in toks1 and "for" not in toks1, "Stopwords must be stripped."
    print("  ✓ Check 1: Text sanitization and stop-word filtering verified.")
    
    # Test 2: Vocabulary bounds
    assert len(vectorizer.vocabulary_) > 100, "Vocabulary must contain over 100 domain tokens."
    print("  ✓ Check 2: Vocabulary dimension verified.")
    
    # Test 3: Cosine Similarity mathematical boundary conditions
    # Identical text must yield cosine similarity == 1.0 (within numerical float tolerance)
    exact_q = faq_df.loc[0, "full_text"]
    q_vec = vectorizer.transform([exact_q])
    sim_exact = cosine_similarity(q_vec, doc_matrix[0:1])[0][0]
    np.testing.assert_allclose(sim_exact, 1.0, rtol=1e-4, err_msg="Self-similarity of identical text must equal 1.0")
    print("  ✓ Check 3: Identity vector cosine projection (sim == 1.0) verified.")
    
    # Test 4: Orthogonal / Zero-overlap queries must yield similarity == 0.0
    gibberish_q = "xyzqwk123 nonexistingtoken999"
    res_gibberish = search_campus_faq(gibberish_q, vectorizer, doc_matrix, faq_df)
    assert res_gibberish[0]["cosine_score"] == 0.0, "Disjoint vocabulary query must yield exact 0.0 cosine similarity."
    print("  ✓ Check 4: Orthogonal disjoint vocabulary handling (sim == 0.0) verified.")
    
    # Test 5: Fallback trigger thresholding
    fb_res = search_with_fallback(gibberish_q, vectorizer, doc_matrix, faq_df, confidence_threshold=0.25)
    assert fb_res["status"] == "FALLBACK_TRIGGERED", "Fallback must trigger on zero-score queries."
    print("  ✓ Check 5: Automated fallback threshold logic verified.")
    
    print("\n" + "=" * 80)
    print("🎉 ALL SELF-CHECK UNIT TESTS PASSED WITH ZERO ERRORS!")
    print("=" * 80)

run_comprehensive_self_check()
